# Notebook 01 — Exploração de Dados e Análise de Latência

## Aula 01: Fundamentos de Latência e Performance em ML

---

### 🎯 Objetivos deste Notebook

1. Carregar e explorar o dataset sintético de classificação
2. Realizar análise exploratória (EDA) com foco em características que impactam a latência
3. Entender a estrutura dos dados e como dimensionalidade afeta performance
4. Medir a latência das operações básicas de I/O e pré-processamento

### 📖 Conceito Teórico

Conforme Crankshaw et al. (2017) — *Clipper: A Low-Latency Online Prediction Serving System*:

> A latência total de um sistema de ML em produção é composta por múltiplos componentes:
> $$L_{total} = L_{rede} + L_{pre} + L_{inferência} + L_{pos} + L_{serialização}$$
>
> Neste notebook, focamos em **$L_{pre}$** — a latência de pré-processamento, que inclui
> carregamento de dados, limpeza e normalização de features.

---

In [ ]:
# Imports necessários
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Adicionar diretório raiz ao path para imports dos módulos src/
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor
from src.utils import setup_output_dirs

# Configurações de visualização
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Seed para reprodutibilidade
np.random.seed(42)

# Criar diretórios de output
dirs = setup_output_dirs(str(Path.cwd().parent))
print("Ambiente configurado com sucesso!")

## 1. Geração e Carregamento do Dataset

O dataset é **sintético**, gerado com `sklearn.datasets.make_classification`.
Contém 10.000 instâncias com 15 features (10 informativas, 2 redundantes, 3 ruidosas).

Vamos medir a latência de geração e carregamento — essa é a primeira componente
da latência de pré-processamento ($L_{pre}$).

In [ ]:
# Gerar dataset sintético (se não existir)
dataset_path = str(Path.cwd().parent / "data" / "raw" / "dataset.csv")

if not Path(dataset_path).exists():
    print("Gerando dataset sintético...")
    start = time.perf_counter()
    df = DataPreprocessor.generate_synthetic_dataset(
        output_path=dataset_path,
        n_samples=10_000,
        n_features=15,
        n_informative=10,
        random_state=42,
    )
    gen_time = time.perf_counter() - start
    print(f"Dataset gerado em {gen_time*1000:.2f} ms")
else:
    print("Dataset já existe, carregando...")

# Carregar com medição de latência
preprocessor = DataPreprocessor(random_state=42)
df = preprocessor.load_data(dataset_path)

print(f"\nDataset shape: {df.shape}")
print(f"Colunas: {list(df.columns)}")

## 2. Visão Geral do Dataset

A compreensão da estrutura dos dados é essencial antes de qualquer análise de performance.
O número de features, a escala dos dados e a presença de correlações impactam
diretamente a complexidade computacional dos modelos.

In [ ]:
# Estatísticas descritivas
print("=" * 60)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 60)
print(f"\nShape: {df.shape[0]} instâncias x {df.shape[1]} colunas")
print(f"Features: {df.shape[1] - 1}")
print(f"\nDistribuição do Target:")
print(df["target"].value_counts().to_string())
print(f"\nBalanceamento: {df['target'].value_counts(normalize=True).to_dict()}")
print(f"\nNulos por coluna: {df.isnull().sum().sum()} no total")
print(f"Duplicatas: {df.duplicated().sum()} linhas")

df.describe().round(3)

## 3. Análise de Distribuições

A distribuição das features influencia a eficácia da normalização ($z = \frac{x - \mu}{\sigma}$)
e, consequentemente, a velocidade de convergência de algoritmos como Logistic Regression e SVM.

In [ ]:
# Distribuição das features (primeiras 6)
feature_cols = [c for c in df.columns if c != "target"]

fig, axes = plt.subplots(3, 5, figsize=(20, 10))
for idx, col in enumerate(feature_cols):
    ax = axes[idx // 5, idx % 5]
    for label in [0, 1]:
        subset = df[df["target"] == label][col]
        ax.hist(subset, bins=30, alpha=0.5, label=f"Classe {label}", density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle("Distribuição das Features por Classe", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(str(dirs["figures"] / "01_feature_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Análise de Correlações

Features altamente correlacionadas (redundantes) aumentam a dimensionalidade sem
adicionar informação. Isso impacta:
- **Latência de treinamento**: Mais features = mais operações matriciais
- **Latência de inferência**: Especialmente em SVM (complexidade O(n_sv · d))
- **Memória**: Mais features = mais memória para armazenar pesos

In [ ]:
# Matriz de correlação
fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df[feature_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    square=True,
)
ax.set_title("Matriz de Correlação entre Features", fontsize=14)
fig.savefig(str(dirs["figures"] / "01_correlation_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

# Identificar features altamente correlacionadas
high_corr = []
for i in range(len(corr_matrix)):
    for j in range(i + 1, len(corr_matrix)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr.append(
                (corr_matrix.index[i], corr_matrix.columns[j], corr_matrix.iloc[i, j])
            )

if high_corr:
    print("Features altamente correlacionadas (|r| > 0.7):")
    for f1, f2, r in high_corr:
        print(f"  {f1} <-> {f2}: r = {r:.3f}")
else:
    print("Nenhum par com correlação > 0.7 encontrado.")

## 5. Análise de Separabilidade entre Classes

A separabilidade das classes influencia:
- A complexidade do modelo necessário (modelos mais complexos = maior latência)
- O número de support vectors em SVM (mais SVs = inferência mais lenta)
- A profundidade necessária em árvores de decisão

In [ ]:
# PCA 2D para visualização de separabilidade
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(df[feature_cols].values)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=df["target"].values,
    cmap="RdYlBu",
    alpha=0.3,
    s=10,
)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variância)", fontsize=12)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variância)", fontsize=12)
ax.set_title("Projeção PCA 2D — Separabilidade entre Classes", fontsize=14)
plt.colorbar(scatter, label="Classe")
fig.savefig(str(dirs["figures"] / "01_pca_separability.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Variância explicada pelas 2 primeiras componentes: "
      f"{sum(pca.explained_variance_ratio_[:2])*100:.1f}%")

## 6. Medição de Latência do Pré-processamento

Conforme o framework de decomposição de latência:

$$L_{pre} = L_{IO} + L_{limpeza} + L_{normalização}$$

Vamos medir cada componente individualmente para identificar possíveis gargalos.

In [ ]:
# Pipeline completo de pré-processamento com medição
preprocessor = DataPreprocessor(random_state=42)

# Etapa 1: Carregar dados
df = preprocessor.load_data(dataset_path)

# Etapa 2: Limpar dados
df_clean = preprocessor.clean_data(df)

# Etapa 3: Preparar features (normalização)
X, y = preprocessor.prepare_features(df_clean)

# Etapa 4: Split dos dados
splits = preprocessor.split_data(X, y)

# Relatório de latência
latency_report = preprocessor.get_latency_report()
print("=" * 50)
print("RELATÓRIO DE LATÊNCIA DO PRÉ-PROCESSAMENTO")
print("=" * 50)
print(latency_report.to_string(index=False))
print(f"\nLatência total: {latency_report['latency_ms'].sum():.2f} ms")

In [ ]:
# Visualização da decomposição de latência
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colors = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12"]
axes[0].barh(
    latency_report["operation"],
    latency_report["latency_ms"],
    color=colors[: len(latency_report)],
    alpha=0.8,
)
axes[0].set_xlabel("Latência (ms)", fontsize=12)
axes[0].set_title("Latência por Etapa do Pré-processamento", fontsize=13)
axes[0].grid(True, alpha=0.3, axis="x")

# Gráfico de pizza (proporção)
axes[1].pie(
    latency_report["latency_ms"],
    labels=latency_report["operation"],
    autopct="%1.1f%%",
    colors=colors[: len(latency_report)],
    startangle=90,
)
axes[1].set_title("Distribuição da Latência de Pré-processamento", fontsize=13)

fig.suptitle("Decomposição de L_pre", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(str(dirs["figures"] / "01_preprocessing_latency.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Impacto do Tamanho dos Dados na Latência de I/O

A relação entre o volume de dados e a latência de I/O é crucial para
dimensionar sistemas de ML. Vamos medir como o tempo de leitura escala
com o número de instâncias.

In [ ]:
# Medir latência de leitura para diferentes tamanhos
import tempfile

sizes = [100, 500, 1000, 2000, 5000, 10000]
io_latencies = []

for size in sizes:
    # Criar CSV temporário com subset
    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w") as tmp:
        df.head(size).to_csv(tmp, index=False)
        tmp_path = tmp.name

    # Medir latência de leitura (5 iterações, média)
    times = []
    for _ in range(5):
        start = time.perf_counter()
        _ = pd.read_csv(tmp_path)
        times.append((time.perf_counter() - start) * 1000)

    io_latencies.append({"n_samples": size, "latency_ms": np.mean(times)})
    Path(tmp_path).unlink()  # Limpar temporário

io_df = pd.DataFrame(io_latencies)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(io_df["n_samples"], io_df["latency_ms"], "o-", linewidth=2, color="steelblue")
ax.set_xlabel("Número de Instâncias", fontsize=12)
ax.set_ylabel("Latência de I/O (ms)", fontsize=12)
ax.set_title("Escalabilidade da Latência de I/O com Volume de Dados", fontsize=14)
ax.grid(True, alpha=0.3)
fig.savefig(str(dirs["figures"] / "01_io_latency_scaling.png"), dpi=150, bbox_inches="tight")
plt.show()

print(io_df.to_string(index=False))

## 8. Análise de Dimensionalidade vs. Performance

A **maldição da dimensionalidade** é um conceito central em ML.
Mais features significam:
- Maior latência de normalização: O(n · d)
- Maior latência de inferência para modelos lineares: O(d)
- Maior número de support vectors em SVM

In [ ]:
# Medir latência de normalização para diferentes números de features
from sklearn.preprocessing import StandardScaler

n_features_list = [2, 5, 10, 15]
norm_latencies = []

for n_feat in n_features_list:
    X_subset = df[feature_cols[:n_feat]].values
    scaler = StandardScaler()

    times = []
    for _ in range(20):
        start = time.perf_counter()
        scaler.fit_transform(X_subset)
        times.append((time.perf_counter() - start) * 1000)

    norm_latencies.append({
        "n_features": n_feat,
        "mean_ms": np.mean(times),
        "std_ms": np.std(times),
    })

norm_df = pd.DataFrame(norm_latencies)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    norm_df["n_features"].astype(str),
    norm_df["mean_ms"],
    yerr=norm_df["std_ms"],
    capsize=5,
    color="steelblue",
    alpha=0.8,
)
ax.set_xlabel("Número de Features", fontsize=12)
ax.set_ylabel("Latência de Normalização (ms)", fontsize=12)
ax.set_title("Impacto da Dimensionalidade na Latência de Normalização", fontsize=14)
ax.grid(True, alpha=0.3, axis="y")
fig.savefig(str(dirs["figures"] / "01_dimensionality_latency.png"), dpi=150, bbox_inches="tight")
plt.show()

print(norm_df.to_string(index=False))

## 9. Resumo dos Dados Preparados

Os dados processados estão prontos para serem utilizados no **Notebook 02 (Treinamento)**.

In [ ]:
# Resumo dos splits
print("=" * 50)
print("RESUMO DOS DADOS PREPARADOS")
print("=" * 50)
for name, arr in splits.items():
    print(f"  {name:10s}: shape = {arr.shape}")

# Salvar dados processados para uso nos próximos notebooks
processed_dir = Path.cwd().parent / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

for name, arr in splits.items():
    np.save(str(processed_dir / f"{name}.npy"), arr)

print(f"\nDados salvos em: {processed_dir}")
print("Arquivos: ", [f.name for f in processed_dir.glob("*.npy")])

## 📋 Resumo

### O que aprendemos neste notebook:

1. **Estrutura dos dados**: 10K instâncias, 15 features (10 info, 2 redundantes, 3 ruído)
2. **Balanceamento**: ~50/50 entre classes (adequado para foco em latência)
3. **Latência de pré-processamento**: Decomposição em I/O, limpeza e normalização
4. **Escalabilidade**: Como o volume de dados e dimensionalidade afetam a latência

### 🔗 Próximo Notebook

No **Notebook 02 — Treinamento**, vamos:
- Treinar múltiplos algoritmos (Logistic Regression, Random Forest, Gradient Boosting, SVM)
- Medir latência de treinamento e inferência para cada um
- Comparar complexidade computacional teórica vs. empírica

### 📚 Referências

- Crankshaw, D. et al. (2017). *Clipper: A Low-Latency Online Prediction Serving System*. NSDI '17.
- Sculley, D. et al. (2015). *Hidden Technical Debt in ML Systems*. NeurIPS '15.